# Taller 2 v3: SQL, Pandas y Limpieza de Datos
### Mentoría Data Science

Lo que aprenderás en este notebook:
1.  **SQL fundamental** — SELECT, WHERE, LIKE, BETWEEN, JOIN, GROUP BY, HAVING, ORDER BY
2.  **SQL + pandas/matplotlib** — integrar consultas con análisis y visualización
3.  **SQL avanzado** — CASE WHEN, CTEs (WITH), funciones de fecha, subconsultas
4.  **Limpieza de datos** — NULLs, tipos, outliers, consistencia
5.  **20 challenges** con starter code para practicar

**Base de datos:** ClassicModels — empresa ficticia de venta de modelos de coches a escala.
8 tablas: `customers`, `orders`, `orderdetails`, `products`, `productlines`, `employees`, `offices`, `payments`

## Configuracion inicial

Con Docker **no hay que cambiar nada** — la ruta a la base de datos es automatica.

Si ejecutas el notebook en local sin Docker, exporta la variable de entorno antes de abrir Jupyter:




In [ ]:
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import os
# Docker: ruta dentro del contenedor (no tocar)
# Local:  variable de entorno DB_PATH o edita la linea
db_path = os.environ.get('DB_PATH', '/home/jovyan/work/data/classic.db')

conn = sqlite3.connect(db_path)
cursor = conn.cursor()
print(" Conexión establecida.")
print("Ejecuta conn.close() al terminar el notebook para liberar el archivo.")

##  Paso 0 — Explorar el esquema antes de consultar

Antes de escribir cualquier consulta, es imprescindible conocer qué tablas existen
y qué columnas tiene cada una. Este es siempre el primer paso.

In [ ]:
# Listar todas las tablas
cursor.execute("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name;")
tablas = [row[0] for row in cursor.fetchall()]
print(f"Tablas en classic.db ({len(tablas)}):", tablas, "\n")

# Estructura de cada tabla
for tabla in tablas:
    cursor.execute(f"PRAGMA table_info({tabla});")
    cols = cursor.fetchall()
    print(f" {tabla}")
    for c in cols:
        pk   = " " if c[5] else ""
        null = "" if c[3] else " (nullable)"
        print(f"   {c[1]:<35} {c[2]:<10}{pk}{null}")
    print()

##  Análisis — Exploración del esquema

###  Qué hace este bloque
1. **`sqlite_master`** es la tabla interna de metadatos de SQLite. Registra todos los objetos (tablas, vistas, índices). La consulta `WHERE type='table'` es el equivalente de `SHOW TABLES` en MySQL.
2. **`PRAGMA table_info(tabla)`** devuelve una fila por columna: posición, nombre, tipo de dato, restricción NOT NULL, valor por defecto, y si es clave primaria (`pk=1`).
3. El bucle recorre cada tabla e imprime su estructura con formato legible.

###  Conceptos clave
- **Claves de unión**: busca columnas que terminan en `Number` o `Code`. Son las claves foráneas que conectan tablas. `customerNumber` aparece en `customers`, `orders` y `payments`.
- **Tipos en SQLite**: TEXT, INTEGER, REAL. SQLite es de tipado dinámico — una columna TEXT puede guardar números, lo que puede causar errores en comparaciones o cálculos.
- **Nullable**: columnas sin NOT NULL pueden contener `NULL`, lo que afecta al comportamiento de `COUNT`, `WHERE` y agregaciones.

###  Diagrama de relaciones
```
customers ─(customerNumber)─► orders ─(orderNumber)─► orderdetails ─(productCode)─► products
                                                                                          │
employees ─(officeCode)─► offices                    productlines ◄─(productLine)────────┘
customers ─(customerNumber)─► payments
```

---
#  Sección 1: Consultas SQL fundamentales

Patrón de cada ejercicio: **Prompt → Código SQL → Análisis del código → Variante mejorada**

### Ejercicio 1 — Número de pedidos por cliente

**Prompt:**
*"Escribe un script en Python que obtenga el número total de pedidos por cliente.
Muestra el nombre del cliente y la cantidad, ordenado de mayor a menor."*

In [ ]:
# Tu consulta aqui
# Tablas: customers (c), orders (o)
# Une por: customerNumber
# Muestra: nombre del cliente y total de pedidos
# Ordena: de mayor a menor

cursor.execute("""
    -- escribe tu SELECT aqui
""")
for fila in cursor.fetchall()[:10]:
    print(fila)


##  Análisis — Ejercicio 1: Número de pedidos por cliente

###  Qué hace este bloque
1. `INNER JOIN orders o ON c.customerNumber = o.customerNumber` — une cada fila de `customers` con sus pedidos. Los alias `c` y `o` acortan los nombres de tabla (convención estándar en SQL).
2. `COUNT(o.orderNumber)` — cuenta cuántos pedidos hay por cliente. Usar la clave primaria garantiza que no haya NULLs que COUNT pudiera ignorar.
3. `GROUP BY c.customerName` — colapsa todas las filas del mismo cliente en una sola. **Regla de oro**: siempre que mezcles una función de agregación (`COUNT`, `SUM`, `AVG`) con columnas normales en el SELECT, `GROUP BY` es obligatorio.
4. `ORDER BY total_pedidos DESC` — usa el alias definido en SELECT. El alias funciona en `ORDER BY` pero **no en `WHERE`** (en WHERE el alias aún no existe).

###  INNER JOIN vs LEFT JOIN
- **INNER JOIN**: solo incluye los ~98 clientes que han hecho al menos un pedido. Los ~24 sin pedidos desaparecen del resultado.
- **LEFT JOIN**: incluiría todos los clientes, con `total_pedidos = 0` para los que no han pedido nada.

###  Prueba esto
Cambia `INNER JOIN` por `LEFT JOIN` y añade `WHERE o.orderNumber IS NULL` para encontrar clientes sin ningún pedido.

### Ejercicio 2 — Top 5 productos más vendidos

**Prompt:**
*"Obtén los 5 productos más vendidos en cantidad total de unidades. Muestra el nombre del producto,
la cantidad total vendida y el total facturado."*

In [ ]:
# Tu consulta aqui
# Tablas: products (p), orderdetails (od)
# Une por: productCode
# Muestra: nombre, unidades totales vendidas, total facturado
# Limita: top 5

cursor.execute("""
    -- escribe tu SELECT aqui
""")
for fila in cursor.fetchall():
    print(fila)


##  Análisis — Ejercicio 2: Top 5 productos más vendidos

###  Qué hace este bloque
1. `SUM(od.quantityOrdered)` — suma todas las unidades pedidas de cada producto a lo largo de todos los pedidos. Un producto puede aparecer en muchos `orderdetails` distintos (uno por cada pedido que lo incluya).
2. `SUM(od.quantityOrdered * od.priceEach)` — aritmética directamente en SQL. No necesitas pandas para multiplicar columnas; los operadores aritméticos `+`, `-`, `*`, `/` funcionan dentro de cualquier expresión SQL.
3. `GROUP BY p.productCode, p.productName` — mejor agrupar por la clave primaria (`productCode`) que solo por nombre, para evitar colisiones si dos productos tuvieran el mismo nombre.
4. `LIMIT 5` — devuelve solo las 5 primeras filas tras el `ORDER BY`. Siempre va al final de la consulta.

###  ¿Por qué JOIN aquí?
Los nombres de producto están en `products`, pero las cantidades y precios de venta están en `orderdetails`. El JOIN une ambas tablas por `productCode` para combinar la información de ambas fuentes.

###  Observación sobre los datos
El Ferrari 360 Spider red vendió 1808 unidades ($276k), pero el 2nd producto vendió menos de la mitad ($102k). Hay diferencias de precio unitario importantes. Explóralas calculando el margen.

### Ejercicio 3 — Ventas totales por país

**Prompt:**
*"Obtén las ventas totales por país. Muestra el país y el total facturado, ordenado de mayor a menor."*

In [ ]:
# Tu consulta aqui
# Tablas: customers (c), orders (o), orderdetails (od)
# Muestra: pais y ventas totales
# Ordena: de mayor a menor

cursor.execute("""
    -- escribe tu SELECT aqui
""")
for fila in cursor.fetchall():
    print(fila)


##  Análisis — Ejercicio 3: Ventas por país

###  Qué hace este bloque
1. **3 tablas en cadena**: `customers → orders → orderdetails`.
   - Paso 1: `customers INNER JOIN orders` → combina cada cliente con sus pedidos.
   - Paso 2: resultado `INNER JOIN orderdetails` → añade el detalle de productos y precios.
2. El precio de venta (`priceEach`) está en `orderdetails`, no en `products`. Refleja el precio real de la transacción, que puede diferir del precio catálogo (`MSRP`).
3. El país (`c.country`) viene de `customers`, pero los importes vienen de `orderdetails`. El JOIN actúa como puente a través de las 3 tablas.

###  ¿Por qué no necesitamos la tabla `products`?
Porque no usamos ningún dato de `products` (nombre, línea, etc.). Solo añadimos tablas al JOIN si necesitamos columnas de ellas.

###  USA domina con $3.27M — ¿es un problema de concentración?
Añade `COUNT(DISTINCT c.customerNumber) AS num_clientes` al SELECT y divide `total_ventas / num_clientes` para ver la venta media por cliente en cada país.

### Ejercicio 4 — Inventario por línea de producto

**Prompt:**
*"Obtén la cantidad total en inventario para cada línea de producto, ordenada de mayor a menor."*

In [ ]:
# Tu consulta aqui
# Tabla: products
# Muestra: linea de producto, numero de referencias, total inventario

cursor.execute("""
    -- escribe tu SELECT aqui
""")
for fila in cursor.fetchall():
    print(fila)


##  Análisis — Ejercicio 4: Inventario por categoría

###  Qué hace este bloque
1. **Una sola tabla, sin JOIN** — toda la información necesaria está en `products`. Cuando el análisis no requiere datos de otras tablas, el JOIN no es necesario ni conveniente.
2. `COUNT(productCode)` — cuenta el número de referencias (SKUs) por línea. `productCode` es la clave primaria, nunca NULL, así que `COUNT` siempre devuelve el número correcto de filas.
3. `ROUND(AVG(buyPrice), 2)` — calcula el precio medio de compra y lo redondea a 2 decimales. Las funciones de agregación se pueden combinar y anidar con funciones escalares como `ROUND`, `UPPER`, `strftime`, etc.

###  Interpretación
- **Classic Cars** domina con 219k unidades — casi el doble que Vintage Cars.
- **Trains** tiene solo 3 referencias y 16k unidades — línea de nicho.
- El precio de compra más alto no garantiza el mayor inventario en valor. Prueba: `SUM(quantityInStock * buyPrice)` para calcular el inventario en valor monetario.

### Ejercicio 5 — Clientes sin pedidos recientes *(bug corregido)*

**Prompt:**
*"Lista los clientes que no han realizado pedidos después de junio de 2004, con la fecha de su último pedido."*

>  **Bug del notebook anterior**: el código original usaba `datetime.now()` como fecha de corte,
> lo que producía que **todos** los clientes aparecieran como inactivos (los datos son de 2003–2005).
> Se usa una fecha fija dentro del rango real de los datos.

In [ ]:
# Fecha de corte — no uses datetime.now() (los datos son de 2003-2005)
FECHA_CORTE = "2004-06-01"

# Tu consulta aqui
# Pista: LEFT JOIN + MAX(orderDate) + HAVING
cursor.execute("""
    -- escribe tu SELECT aqui
""", (FECHA_CORTE,))
for fila in cursor.fetchall()[:10]:
    print(fila)


##  Análisis — Ejercicio 5: Clientes sin pedidos recientes

###  Qué hace este bloque
1. **LEFT JOIN** — incluye todos los clientes aunque no tengan pedidos. Para clientes sin pedidos en `orders`, todas las columnas de `orders` son NULL después del JOIN.
2. `MAX(o.orderDate)` — obtiene la fecha del último pedido. Para clientes sin pedidos, `MAX(NULL)` devuelve `NULL`.
3. **`HAVING` en lugar de `WHERE`** — HAVING filtra *después* de la agregación (tras GROUP BY). `WHERE` filtra antes de agrupar y no puede referenciar alias de agregación como `ultimo_pedido`.
4. `HAVING ultimo_pedido IS NULL OR ultimo_pedido < ?` — el `?` es un parámetro enlazado. Es la forma correcta de pasar valores variables a SQL en Python: evita inyección SQL y gestiona el escapado automáticamente.

###  LEFT JOIN + IS NULL = patrón anti-join
`LEFT JOIN ... WHERE clave_foranea IS NULL` (o en HAVING) es la forma estándar de encontrar registros en una tabla sin correspondencia en otra. Es más claro que `NOT IN (subconsulta)` y más eficiente en bases de datos grandes.

###  Por qué `datetime.now()` era un bug
Con `datetime.now() - timedelta(days=180)` el umbral sería ~octubre 2025. Como todos los pedidos son anteriores a 2005, todos los clientes aparecerían como inactivos. La consulta devolvería resultados "correctos" sintácticamente pero semánticamente inútiles.

### Ejercicio 6 — Filtros: WHERE, LIKE y BETWEEN *(nuevo)*

**Prompt:**
*"Encuentra los productos con 'Ferrari' en el nombre y un precio de compra entre $50 y $100.
Muestra también el stock disponible."*

In [ ]:
# Tu consulta aqui
# Pista: LIKE con comodin %, BETWEEN para rango de precios
cursor.execute("""
    -- escribe tu SELECT aqui
""")
for fila in cursor.fetchall():
    print(fila)


##  Análisis — Ejercicio 6: Filtros WHERE, LIKE, BETWEEN

###  Qué hace este bloque
1. `LIKE '%Ferrari%'` — `%` es comodín para cualquier número de caracteres. `%Ferrari%` encuentra el término en cualquier posición. `Ferrari%` solo encontraría nombres que empiezan por Ferrari. `_` (guión bajo) es comodín para exactamente 1 carácter.
2. `BETWEEN 50 AND 100` — equivale a `buyPrice >= 50 AND buyPrice <= 100`. Los límites son **inclusivos** en ambos extremos.
3. `AND` combina condiciones: ambas deben cumplirse. Con `OR` bastaría que una se cumpla.
4. `IN ('Classic Cars', 'Vintage Cars', 'Motorcycles')` — equivale a tres condiciones `OR`. Más legible para listas de valores exactos.

###  Tabla resumen de operadores de filtro
| Operador | Uso | Ejemplo |
|---|---|---|
| `=` | Igualdad exacta | `country = 'Spain'` |
| `<>` / `!=` | Distinto | `status <> 'Cancelled'` |
| `LIKE '%x%'` | Contiene | `productName LIKE '%Ford%'` |
| `BETWEEN a AND b` | Rango inclusivo | `amount BETWEEN 1000 AND 5000` |
| `IN (lista)` | En lista | `country IN ('USA', 'Spain')` |
| `IS NULL` | Sin valor | `shippedDate IS NULL` |
| `IS NOT NULL` | Tiene valor | `comments IS NOT NULL` |

### Ejercicio 7 — Funciones de fecha: evolución temporal *(nuevo)*

**Prompt:**
*"Analiza las ventas por año y por mes para identificar patrones estacionales."*

In [ ]:
# Tu consulta aqui
# Pista: strftime("%Y", orderDate) para extraer el anio
cursor.execute("""
    -- ventas por anio
""")
for fila in cursor.fetchall():
    print(fila)

cursor.execute("""
    -- ventas por mes
""")
for fila in cursor.fetchall():
    print(fila)


##  Análisis — Ejercicio 7: Funciones de fecha

###  Qué hace este bloque
1. `strftime('%Y', o.orderDate)` — extrae el año de una fecha en texto (formato 'YYYY-MM-DD'). `strftime` significa *"string format time"*.
   - `'%Y'` → año completo (2003, 2004, 2005)
   - `'%m'` → mes con cero inicial (01–12)
   - `'%d'` → día
   - `'%Y-%m'` → año-mes (2003-01)
2. `COUNT(DISTINCT o.orderNumber)` — sin DISTINCT, un pedido con 5 líneas en `orderdetails` se contaría 5 veces.
3. Las fechas en SQLite se almacenan como TEXT. `strftime` permite extraer componentes para agrupar sin convertir el tipo.

###  Interpretación de los datos
- **2005 parece muy bajo**: los datos terminan en mayo 2005, no es que las ventas cayeran.
- **Noviembre-diciembre son los meses fuertes**: patrón típico de ventas de productos de regalo/ocio antes de Navidad.
- Para tendencia mes a año, usa `strftime('%Y-%m', orderDate)` en el `GROUP BY`.

###  ¿Y los datos de pandas?
Una vez cargado el resultado en un DataFrame, puedes convertir `anio` a entero con `df['anio'].astype(int)` para graficar correctamente el eje X.

---
#  Sección 2: SQL + Pandas

### ¿Por qué combinar SQL con pandas?

| SQL es mejor para... | Pandas es mejor para... |
|---|---|
| Filtrar y unir tablas grandes en la fuente | Columnas calculadas derivadas |
| Agregaciones sobre millones de filas | Visualizaciones con matplotlib |
| Lógica relacional compleja (JOINs, CTEs) | Estadísticas descriptivas, reshape |

**Patrón estándar:**
```python
df = pd.read_sql("SELECT ...", conn)   # SQL extrae y filtra
df['nueva_col'] = df['a'] / df['b']    # Pandas enriquece
df.plot(...)                           # Matplotlib visualiza
```

### Ejercicio 8 — Productos más vendidos con análisis de margen

**Prompt:**
*"Obtén los productos con su facturación total. Usa pandas para calcular el precio medio de venta
y compararlo con el coste de compra."*

In [ ]:
df_productos = pd.read_sql("""
    -- escribe tu SELECT aqui
    -- incluye: productName, productLine, buyPrice, suma unidades, suma facturado
""", conn)

# Calcula en pandas:
# df_productos["precio_medio_venta"] = ???
# df_productos["margen_pct"] = ???
print(df_productos.head(10))


##  Análisis — Ejercicio 8: SQL + Pandas para análisis de márgenes

###  División de responsabilidades
1. **SQL hace el trabajo pesado**: extrae datos de 2 tablas, agrega sumas por producto, y pre-selecciona las columnas necesarias. Resultado: una fila por producto.
2. **Pandas añade columnas derivadas** que serían verbosas en SQL puro:
   - `precio_medio_venta`: ingresos totales / unidades vendidas — precio promedio real de la transacción.
   - `margen_pct`: diferencia precio venta - coste, expresada en porcentaje sobre el coste.
3. `pd.read_sql("...", conn)` — combina `cursor.execute` + `pd.DataFrame(cursor.fetchall(), columns=[...])` en una sola línea. Más conciso y limpio.

###  Regla para decidir qué va en SQL y qué en pandas
- **SQL**: todo lo que reduce el número de filas o columnas antes de cargar en memoria (filtros, JOINs, GROUP BY).
- **Pandas**: lo que no reduce filas sino que enriquece (columnas calculadas, visualizaciones, estadísticas sobre el DataFrame ya reducido).

### Ejercicio 9 — Ventas por país con visualización

**Prompt:**
*"Genera un gráfico de barras horizontal con los 10 países de mayor facturación."*

In [ ]:
df_paises = pd.read_sql("""
    -- escribe tu SELECT aqui (top 10 paises por ventas)
""", conn)

# Genera un grafico de barras horizontal
fig, ax = plt.subplots(figsize=(9, 5))
# ax.barh(???)
# ax.set_title("Top 10 paises por facturacion")
plt.tight_layout()
plt.show()


##  Análisis — Ejercicio 9: Gráfico barras horizontal

###  Qué hace este bloque
1. **`LIMIT 10` en SQL** — pedimos solo los 10 mejores directamente, no en pandas. Reduce la transferencia de datos.
2. **`barh`** (horizontal) — más legible que `bar` (vertical) cuando los nombres de categoría son largos.
3. `df_paises['country'][::-1]` — invierte el orden porque `barh` dibuja de abajo a arriba. Sin invertir, el mayor quedaría abajo.
4. `plt.FuncFormatter(lambda x, _: f'${x/1e6:.1f}M')` — formatea el eje X en millones con símbolo de dólar. El segundo argumento `_` es la posición (no usada).

###  Mejoras al gráfico
```python
# Línea de media global
media = df_paises['total_ventas'].mean()
ax.axvline(media, color='red', linestyle='--', alpha=0.7, label=f'Media ${media/1e6:.1f}M')
ax.legend()
```

### Ejercicio 10 — Top 10 clientes por facturación

**Prompt:**
*"Identifica los 10 clientes más importantes por facturación y visualiza qué porcentaje del total representan."*

In [ ]:
# Tu consulta aqui
# Tablas: customers (c), orders (o)
# Une por: customerNumber
# Muestra: nombre del cliente y total de pedidos
# Ordena: de mayor a menor

cursor.execute("""
    -- escribe tu SELECT aqui
""")
for fila in cursor.fetchall()[:10]:
    print(fila)


##  Análisis — Ejercicio 10: Top 10 clientes

###  Qué hace este bloque
1. **Segunda consulta para el total global**: es más legible hacer una consulta separada que calcular el porcentaje dentro de la misma consulta SQL (requeriría una subconsulta o CTE).
2. `df_clientes['total_facturado'] / total_global * 100` — pandas calcula el porcentaje vectorialmente sobre toda la columna en una línea.
3. `[f"{n[:18]}..." if len(n) > 18 else n for n in df_clientes['customerName']]` — list comprehension para truncar nombres largos en el eje X del gráfico.

###  Interpretación: riesgo de concentración
- Los dos primeros clientes (Euro+ Shopping Channel y Mini Gifts Distributors) representan ~18% del total.
- Alta dependencia de pocos clientes es un riesgo de negocio. El análisis de concentración es un ejercicio estándar en Data Science aplicado a ventas.

---
#  Sección 3: SQL Avanzado

## Conceptos cubiertos
- **CASE WHEN**: lógica condicional dentro de SQL (equivale a `if/elif/else` de Python)
- **CTE (WITH)**: consultas auxiliares con nombre — mejoran legibilidad y permiten reutilizar resultados
- **Subconsultas**: consultas anidadas dentro de WHERE, SELECT o FROM

### SQL Avanzado 1 — CASE WHEN: clasificar clientes por segmento

**Prompt:**
*"Clasifica cada cliente en 'Premium' (>$100k), 'Estándar' ($50k–$100k) o 'Básico' (<$50k)
según su facturación total. Muestra cuántos clientes hay en cada segmento."*

In [ ]:
df_segmentos = pd.read_sql("""
    -- Usa un CTE (WITH ventas_cliente AS (...)) para calcular el total por cliente
    -- Luego usa CASE WHEN para clasificar en Premium / Estandar / Basico
""", conn)
print(df_segmentos["segmento"].value_counts())


##  Análisis — CASE WHEN y CTEs

###  Qué hace este bloque

**CTE — `WITH ventas_cliente AS (...)`**
1. Define una "tabla temporal con nombre" que solo existe durante la ejecución de la consulta.
2. `ventas_cliente` calcula el total facturado por cliente (requiere JOINs y GROUP BY).
3. La consulta principal hace referencia a `ventas_cliente` como si fuera una tabla real.
4. **Sin CTE**, el CASE WHEN necesitaría anidar toda esa subconsulta dentro del FROM, lo que dificulta la lectura.

**CASE WHEN**
```sql
CASE
    WHEN condicion1 THEN valor1
    WHEN condicion2 THEN valor2
    ELSE valor_por_defecto
END
```
- Las condiciones se evalúan **en orden**: si `total_facturado >= 100000`, no se evalúa la segunda condición.
- `ELSE` captura todos los casos que no cumplen ninguna condición anterior.
- Se puede usar en SELECT, ORDER BY, y como argumento de funciones de agregación (`COUNT(CASE WHEN ...)`).

###  Por qué CTE + CASE WHEN juntos
Sin la CTE, el `CASE WHEN` tendría que referenciar `SUM(od.quantityOrdered * od.priceEach)` directamente dentro del mismo SELECT, lo cual no está permitido en SQL (no se puede usar un alias de agregación en el mismo nivel de SELECT donde se define).

### SQL Avanzado 2 — Subconsultas: clientes que nunca han comprado

**Prompt:**
*"Encuentra todos los clientes que no aparecen en la tabla de pedidos."*

In [ ]:
# Metodo 1: NOT IN
cursor.execute("""
    -- escribe tu SELECT: clientes cuyo customerNumber NO esta en orders
""")
print(f"Clientes sin pedidos: {len(cursor.fetchall())}")


##  Análisis — Subconsultas NOT IN vs LEFT JOIN

###  Qué hace este bloque
1. `WHERE customerNumber NOT IN (SELECT DISTINCT customerNumber FROM orders)` — la subconsulta devuelve todos los IDs de cliente que tienen pedidos; el WHERE externo excluye esos IDs.
2. `DISTINCT` en la subconsulta evita duplicados (un cliente puede aparecer en muchos pedidos). Sin DISTINCT el resultado es el mismo pero es menos eficiente.
3. El **Método 2** (LEFT JOIN + IS NULL) es equivalente: el LEFT JOIN produce NULL en las columnas de `orders` para clientes sin pedidos, y el `WHERE o.orderNumber IS NULL` filtra exactamente esos.

###  Tres formas de resolver el mismo problema
```sql
-- 1. NOT IN (más intuitivo)
WHERE customerNumber NOT IN (SELECT customerNumber FROM orders)

-- 2. LEFT JOIN + IS NULL (más eficiente en BDs grandes)
LEFT JOIN orders o ON c.customerNumber = o.customerNumber
WHERE o.customerNumber IS NULL

-- 3. NOT EXISTS (explícito, bueno en BDs grandes)
WHERE NOT EXISTS (SELECT 1 FROM orders o WHERE o.customerNumber = c.customerNumber)
```
En SQLite con datos pequeños los tres son equivalentes. En producción, el NOT IN puede ser lento si la subconsulta devuelve muchos valores.

---
#  Sección 4: Limpieza de Datos

La limpieza es el paso más crítico y costoso de cualquier proyecto de Data Science.
Un dato incorrecto o mal tipado puede invalidar todo un análisis.

**Qué cubre esta sección:**
1. **Diagnóstico de NULLs** — qué columnas tienen valores ausentes y si son errores o estados válidos
2. **Revisión de tipos** — convertir strings a datetime para habilitar cálculos temporales
3. **Detección de outliers** — identificar registros que se alejan del comportamiento normal con el método IQR
4. **Consistencia** — valores duplicados o inconsistentes

> ClassicModels está relativamente limpio, pero el proceso de diagnóstico es el mismo con cualquier dataset real.

### Limpieza 1 — Diagnóstico de NULLs en todas las tablas

In [ ]:
tablas = ["customers", "orders", "orderdetails", "products", "employees", "offices", "payments"]

for tabla in tablas:
    df = pd.read_sql(f"SELECT * FROM {tabla}", conn)
    # Detecta y muestra los NULLs por columna
    # nulls = ???
    # print(nulls[nulls > 0])
    pass


##  Análisis — Diagnóstico de NULLs

###  Qué hace este bloque
1. `pd.read_sql(f"SELECT * FROM {tabla}", conn)` — carga cada tabla como DataFrame para inspeccionarla.
2. `df.isnull().sum()` — cuenta NULLs por columna. Devuelve una Serie con un valor por columna.
3. `nulls[nulls > 0]` — indexación booleana: filtra solo las columnas con algún NULL.
4. El porcentaje `n / len(df) * 100` contextualiza: un NULL en 1% de filas es anecdótico; en 40% es sistemático.

###  Interpretación: no todos los NULLs son errores

| Columna | NULLs | Interpretación |
|---|---|---|
| `orders.shippedDate` | ~14 | Pedidos aún no enviados (status = 'In Process'). NULL válido. |
| `orders.comments` | ~252 | Campo de texto libre opcional. NULL = sin comentarios. Normal. |
| `customers.addressLine2` | ~150 | Segunda línea de dirección — opcional por naturaleza. |
| `employees.reportsTo` | 1 | El CEO no reporta a nadie. Exactamente 1 NULL esperado. |

###  Antes de "limpiar" un NULL, pregunta:
¿Es un *dato faltante* (error) o un *estado válido* (sin enviar, sin superior, sin comentario)?
Eliminar un NULL válido es peor que dejarlo.

### Limpieza 2 — Tipos de datos y conversión de fechas

In [ ]:
df_orders = pd.read_sql("SELECT * FROM orders", conn)

# Muestra los tipos actuales
print(df_orders.dtypes)

# Convierte las columnas de fecha a datetime
# for col in ["orderDate", "requiredDate", "shippedDate"]:
#     df_orders[col] = ???

# Calcula dias entre pedido y envio
# df_orders["dias_hasta_envio"] = ???


##  Análisis — Tipos de datos y conversión

###  Qué hace este bloque
1. SQLite almacena fechas como TEXT (`'2003-01-06'`). Al cargar en pandas, llegan como tipo `object` (string).
2. `pd.to_datetime(col, errors='coerce')` — convierte strings a `datetime64[ns]`. `errors='coerce'` convierte valores inválidos a `NaT` (*Not a Time*) en lugar de lanzar error.
3. Una vez convertidas a datetime, podemos restar fechas: el resultado es un `Timedelta`, y `.dt.days` extrae los días como entero.
4. `df.dropna(subset=['shippedDate'])` — filtra solo pedidos enviados (con fecha de envío). Equivale a `WHERE shippedDate IS NOT NULL` en SQL.

###  Regla sobre tipos
| Tipo de dato | Viene como de SQLite | Conversión pandas necesaria |
|---|---|---|
| Fechas (orderDate) | `object` (string) | `pd.to_datetime()` |
| Precios (buyPrice) | `float64` |  ya correcto |
| Cantidades (quantityInStock) | `int64` |  ya correcto |
| Códigos (productCode) | `object` | mantener como string |

###  Siempre convierte fechas a datetime antes de calcular
Con tipo `object`, operaciones como `.mean()`, `.max()` o la resta de fechas no están disponibles.
Pandas lanzará un `TypeError` si lo intentas.

### Limpieza 3 — Detección de outliers con el método IQR

In [ ]:
df_ventas = pd.read_sql("""
    SELECT od.quantityOrdered, od.priceEach,
           od.quantityOrdered * od.priceEach AS importe_linea
    FROM orderdetails od
""", conn)

# Implementa la deteccion de outliers con IQR
# Q1 = ???
# Q3 = ???
# IQR = ???
# lim_inf = ???
# lim_sup = ???
# outliers = df_ventas[???]
# print(f"Outliers en importe_linea: {len(outliers)}")


##  Análisis — Detección de outliers con IQR

###  Qué hace el método IQR
1. **IQR (Rango Intercuartílico)** = Q3 − Q1. Mide la dispersión del 50% central de los datos, ignorando los extremos.
2. **Límites de Tukey**: valor es outlier si está fuera de `[Q1 − 1.5·IQR, Q3 + 1.5·IQR]`. El factor 1.5 es convencional; usa 3.0 para "outliers extremos".
3. `serie[(serie < lim_inf) | (serie > lim_sup)]` — indexación booleana compuesta. El `|` es el operador OR vectorizado de pandas (no usar `or`).
4. La función `detectar_outliers_iqr` es reutilizable: recibe cualquier Serie y devuelve los límites.

###  IQR vs desviación estándar
El método IQR es más **robusto** que `media ± 3σ` porque Q1 y Q3 no se ven afectados por los propios outliers (que sesgan la media y σ).

###  ¿Qué hacer con los outliers encontrados?
1. **Investigar primero**: ¿son errores o casos legítimos? (Un pedido grande de un cliente premium no es un error).
2. **Excluir**: solo si son errores de captura que afectan tus métricas clave.
3. **Transformar**: aplicar `log(x)` antes de modelado para reducir el efecto de colas largas.
4. **Documentar**: si los excluyes, deja constancia en el notebook de cuántos y por qué.

---
#  Sección 5: 20 Ejercicios de Práctica

Cada ejercicio incluye:
- Descripción del reto
- Conceptos que necesitas aplicar
- Starter code con la estructura base (completa las partes marcadas con `# ???`)

---

### Challenge 1 — Promedio de ventas por producto
**Reto:** Calcula el precio medio de venta de cada producto (total facturado / unidades vendidas). Muestra los 10 con precio medio más alto.
**Conceptos:** `SUM`, `GROUP BY`, división en SQL

In [ ]:
df_ch1 = pd.read_sql("""
    SELECT
        p.productName,
        p.productLine,
        -- ??? suma del importe total
        -- ??? suma de unidades vendidas
        -- ??? calcula precio_medio = total / unidades
    FROM products p
    INNER JOIN orderdetails od ON p.productCode = od.productCode
    GROUP BY p.productCode, p.productName, p.productLine
    ORDER BY precio_medio DESC
    LIMIT 10
""", conn)
# print(df_ch1)

---
### Challenge 2 — Productos que nunca se han pedido
**Reto:** Encuentra productos en el catálogo que no tienen ninguna línea en `orderdetails`.
**Conceptos:** `NOT IN` con subconsulta, o `LEFT JOIN + IS NULL`

In [ ]:
# Starter: completa la subconsulta del NOT IN
cursor.execute("""
    SELECT productName, productLine, quantityInStock
    FROM products
    WHERE productCode NOT IN (
        -- ??? subconsulta que devuelve los productCode que sí aparecen en orderdetails
    )
""")
# resultados = cursor.fetchall()
# print(f"Productos sin pedidos: {len(resultados)}")

---
### Challenge 3 — Clientes con más de 4 pedidos enviados
**Reto:** Lista solo los clientes que tienen más de 4 pedidos con `status = 'Shipped'`.
**Conceptos:** `WHERE`, `GROUP BY`, `HAVING`

In [ ]:
cursor.execute("""
    SELECT
        c.customerName,
        COUNT(o.orderNumber) AS pedidos_enviados
    FROM customers c
    INNER JOIN orders o ON c.customerNumber = o.customerNumber
    WHERE o.status = ???        -- filtrar por estado
    GROUP BY c.customerName
    HAVING pedidos_enviados > ???  -- condición sobre el agregado
    ORDER BY pedidos_enviados DESC;
""")
# for f in cursor.fetchall(): print(f)

---
### Challenge 4 — Pagos recibidos en 2004 por cliente
**Reto:** Muestra el total pagado por cada cliente durante el año 2004.
**Conceptos:** `strftime`, `WHERE`, `SUM`, `GROUP BY`

In [ ]:
df_ch4 = pd.read_sql("""
    SELECT
        c.customerName,
        SUM(p.amount) AS total_pagado
    FROM customers c
    INNER JOIN payments p ON c.customerNumber = p.customerNumber
    WHERE strftime(???, p.paymentDate) = ???   -- filtrar por año 2004
    GROUP BY c.customerName
    ORDER BY total_pagado DESC
""", conn)
# print(df_ch4.head(10).to_string(index=False))

---
### Challenge 5 — Empleados y sus clientes asignados
**Reto:** Muestra cada empleado de ventas con el número de clientes que tiene asignados y el país de su oficina.
**Conceptos:** JOIN de 3 tablas (employees, customers, offices), `COUNT`

In [ ]:
df_ch5 = pd.read_sql("""
    SELECT
        e.firstName || ' ' || e.lastName AS empleado,
        e.jobTitle,
        o.city AS ciudad_oficina,
        COUNT(???) AS num_clientes    -- cuenta clientes asignados
    FROM employees e
    INNER JOIN offices  o ON e.officeCode       = o.officeCode
    LEFT  JOIN customers c ON ???               -- join con customers por salesRepEmployeeNumber
    GROUP BY e.employeeNumber, empleado, e.jobTitle, o.city
    HAVING e.jobTitle LIKE '%Sales%'
    ORDER BY num_clientes DESC
""", conn)
# print(df_ch5.to_string(index=False))

---
### Challenge 6 — Evolución mensual de ventas 2004 con gráfico de línea
**Reto:** Extrae las ventas totales por mes durante 2004 y visualízalas con un gráfico de línea.
**Conceptos:** `strftime('%Y-%m', ...)`, `pd.read_sql`, `matplotlib.plot`

In [ ]:
df_ch6 = pd.read_sql("""
    SELECT
        strftime('%m', o.orderDate)            AS mes,
        SUM(od.quantityOrdered * od.priceEach) AS total_ventas
    FROM orders o
    INNER JOIN orderdetails od ON o.orderNumber = od.orderNumber
    WHERE strftime('%Y', o.orderDate) = '2004'
    GROUP BY mes
    ORDER BY mes
""", conn)

# Completa el gráfico de línea
fig, ax = plt.subplots(figsize=(9, 4))
meses = ['Ene','Feb','Mar','Abr','May','Jun','Jul','Ago','Sep','Oct','Nov','Dic']
# ax.plot(???, ???, marker='o', color='steelblue')
# ax.set_xticks(range(len(df_ch6)))
# ax.set_xticklabels([meses[int(m)-1] for m in df_ch6['mes']])
# ax.set_title('Ventas por mes — 2004')
# plt.tight_layout(); plt.show()

---
### Challenge 7 — Correlación entre precio y cantidad vendida
**Reto:** ¿Los productos más caros se venden en menor cantidad? Calcula la correlación entre `buyPrice` y `unidades_vendidas` e interpreta el resultado.
**Conceptos:** `pd.read_sql`, `df.corr()`, scatter plot

In [ ]:
df_ch7 = pd.read_sql("""
    SELECT
        p.buyPrice,
        SUM(od.quantityOrdered) AS unidades_vendidas
    FROM products p
    INNER JOIN orderdetails od ON p.productCode = od.productCode
    GROUP BY p.productCode, p.buyPrice
""", conn)

# corr = df_ch7['buyPrice'].corr(df_ch7['unidades_vendidas'])
# print(f"Correlación Pearson: {corr:.3f}")
# df_ch7.plot.scatter(x='buyPrice', y='unidades_vendidas', alpha=0.5)
# plt.title(f'Precio vs Unidades vendidas (r={corr:.3f})')
# plt.show()

---
### Challenge 8 — Segmentación de clientes por frecuencia de compra
**Reto:** Clasifica a los clientes en 3 grupos según su número de pedidos: frecuentes (>5), ocasionales (2–5), nuevos (1).
**Conceptos:** `CASE WHEN`, CTE, `pd.cut` (alternativa en pandas)

In [ ]:
df_ch8 = pd.read_sql("""
    WITH pedidos_cliente AS (
        SELECT customerNumber, COUNT(orderNumber) AS total_pedidos
        FROM orders
        GROUP BY customerNumber
    )
    SELECT
        c.customerName,
        pc.total_pedidos,
        CASE
            WHEN pc.total_pedidos > ???  THEN 'Frecuente'
            WHEN pc.total_pedidos >= ??? THEN 'Ocasional'
            ELSE                              'Nuevo'
        END AS segmento
    FROM customers c
    INNER JOIN pedidos_cliente pc ON c.customerNumber = pc.customerNumber
    ORDER BY pc.total_pedidos DESC
""", conn)
# print(df_ch8['segmento'].value_counts())

---
### Challenge 9 — Productos con stock bajo (< 100 unidades)
**Reto:** Lista los productos con menos de 100 unidades en inventario, ordenados por stock ascendente.
**Conceptos:** `WHERE`, `ORDER BY`

In [ ]:
cursor.execute("""
    SELECT productName, productLine, quantityInStock, buyPrice
    FROM products
    WHERE ???
    ORDER BY ???;
""")
# resultados = cursor.fetchall()
# for f in resultados:
#     print(f"{f[0]:<45} {f[1]:<15} stock: {f[2]:>4}  ${f[3]:.2f}")

---
### Challenge 10 — Deuda de clientes: facturado vs pagado
**Reto:** ¿Qué clientes nos deben dinero? Calcula la diferencia entre lo facturado en pedidos enviados y lo pagado realmente.
**Conceptos:** CTE doble, JOIN, aritmética entre consultas

In [ ]:
df_ch10 = pd.read_sql("""
    WITH facturado AS (
        SELECT
            o.customerNumber,
            SUM(od.quantityOrdered * od.priceEach) AS total_facturado
        FROM orders o
        INNER JOIN orderdetails od ON o.orderNumber = od.orderNumber
        WHERE o.status = 'Shipped'
        GROUP BY o.customerNumber
    ),
    pagado AS (
        SELECT customerNumber, SUM(amount) AS total_pagado
        FROM payments
        GROUP BY customerNumber
    )
    SELECT
        c.customerName,
        COALESCE(f.total_facturado, 0) AS facturado,
        COALESCE(p.total_pagado, 0)    AS pagado,
        COALESCE(f.total_facturado, 0) - COALESCE(p.total_pagado, 0) AS saldo_pendiente
    FROM customers c
    LEFT JOIN facturado f ON c.customerNumber = f.customerNumber
    LEFT JOIN pagado    p ON c.customerNumber = p.customerNumber
    HAVING saldo_pendiente ???  -- completa: clientes con saldo positivo (deben dinero)
    ORDER BY saldo_pendiente DESC
    LIMIT 15
""", conn)
# print(df_ch10.to_string(index=False))

---
### Challenge 11 — Media y mediana de ventas por producto con numpy
**Reto:** Carga todas las ventas por producto y calcula media, mediana y desviación estándar con numpy.
**Conceptos:** `np.mean`, `np.median`, `np.std`

---

### Challenge 12 — Distribución de precios con histograma
**Reto:** Visualiza la distribución del precio de compra (`buyPrice`) de todos los productos con un histograma de 15 bins. Añade líneas verticales para la media y la mediana.
**Conceptos:** `matplotlib.hist`, `ax.axvline`

---

### Challenge 13 — Ventas por trimestre 2003 vs 2004
**Reto:** Compara las ventas por trimestre entre 2003 y 2004 con un gráfico de barras agrupadas.
**Conceptos:** `strftime('%m', ...)`, division entera para trimestre, `pivot_table` en pandas

---

### Challenge 14 — Ranking de ventas por empleado
**Reto:** Cada cliente tiene un `salesRepEmployeeNumber`. Calcula las ventas totales atribuibles a cada comercial.
**Conceptos:** JOIN de 4 tablas: employees → customers → orders → orderdetails

---

### Challenge 15 — Ticket medio por pedido y por cliente
**Reto:** Calcula el importe medio por pedido para cada cliente. ¿Cuáles clientes hacen pedidos más grandes aunque pidan menos veces?
**Conceptos:** CTE, `AVG`, división de sumas

---

### Challenge 16 — Detección de duplicados en pagos
**Reto:** Comprueba si hay registros duplicados en la tabla `payments` (mismo cliente, fecha y monto).
**Conceptos:** `df.duplicated()`, `GROUP BY` con `HAVING COUNT > 1`

---

### Challenge 17 — Normalización z-score de precios
**Reto:** Normaliza el campo `buyPrice` con z-score ((x - media) / std). Identifica los productos con z-score > 2 (más de 2 desviaciones por encima de la media).
**Conceptos:** `(df['col'] - df['col'].mean()) / df['col'].std()`

---

### Challenge 18 — Consistencia de texto: países y ciudades
**Reto:** Comprueba si hay inconsistencias en los nombres de país o ciudad (mayúsculas, espacios extra). Normaliza con pandas.
**Conceptos:** `str.strip()`, `str.title()`, `value_counts()`

---

### Challenge 19 — Primer pedido de cada cliente (CTE con MIN)
**Reto:** Muestra la fecha del primer pedido de cada cliente y cuánto tiempo lleva como cliente hasta hoy (si usas fecha fija 2005-06-01 como "hoy").
**Conceptos:** `MIN(orderDate)`, CTE, funciones de fecha, `julianday()` en SQLite

---

### Challenge 20 — Análisis completo: pipeline SQL → pandas → visualización
**Reto libre:** Elige una pregunta de negocio que no haya sido respondida en el notebook, formula el prompt completo (contexto + datos + objetivo + herramientas + formato), escribe la consulta SQL, cárgala en pandas, calcula al menos una métrica derivada y visualiza el resultado con matplotlib.

---
##  Fin del notebook

Ejecuta la siguiente celda para cerrar la conexión a la base de datos.

In [ ]:
conn.close()
print("Conexión cerrada.")